# Price-Level Robustness Check

The reviewer's concern: our regime-conditional width comparison (Section 5.3 Mechanism, Section 5.4) is
reported in **absolute Sensex/Nifty points**. But the index level itself grew a lot over 2010-2026
(Sensex roughly 17,000 -> 80,000+). If high-volatility windows happen to sit at a higher average price
level than low-volatility windows (just by chance of when volatility spiked historically), then "high-vol
intervals are wider in points" could partly just mean "high-vol windows happened to occur when the index
was already bigger" -- nothing to do with volatility itself.

Fix: redo the same low-vol vs high-vol width comparison using **percentage width**
(`(upper - lower) / origin_price`) instead of raw points. Percentage width cancels out the price-level
effect, so if the regime gap survives here, the paper's mechanism claim is on solid ground.

Note: `forecasts.npy` / `actuals.npy` only store the 30-day-ahead quantiles and outcomes -- not the
price on the day each forecast was made ("origin price"). We don't have that saved anywhere, so this
notebook rebuilds it by re-running the exact same rolling-window loop from `01_load_forecast.ipynb` /
`06_nifty_forecast.ipynb` (same `context_len`, `step`, `start`, `end`), but this time just recording
`prices[i-1]` (the last known price before each forecast) instead of re-running Chronos-2. This is fast
since it's just indexing, no model inference.

In [1]:
import pandas as pd
import numpy as np

## Step 1: Rebuild origin prices for each window

Same window construction as the original forecast notebooks: `context_len=512`, `horizon=30`, `step=10`,
looping `i` from `context_len` to `len(prices) - horizon`. The origin price for window `i` is
`prices[i-1]` -- the last actual close price the model saw before making that forecast.

This must produce arrays with the same length and same row order as `forecasts.npy` / `actuals.npy`,
since those were built with the identical loop. We check the lengths match as a sanity check below.

In [2]:
def rebuild_origin_prices(csv_path, context_len=512, horizon=30, step=10):
    df = pd.read_csv(csv_path, parse_dates=["Date"])
    prices = df["Close"].values

    start = context_len
    end = len(prices) - horizon

    origin_prices = []
    origin_dates = []
    for i in range(start, end, step):
        origin_prices.append(prices[i - 1])   # last known price before this forecast
        origin_dates.append(df["Date"].iloc[i - 1])

    return np.array(origin_prices), np.array(origin_dates)


sensex_origin_prices, sensex_origin_dates = rebuild_origin_prices("../data/sensex_clean.csv")
nifty_origin_prices, nifty_origin_dates = rebuild_origin_prices("../data/nifty_clean.csv")

print("sensex origin prices:", sensex_origin_prices.shape)
print("nifty origin prices:", nifty_origin_prices.shape)

sensex origin prices: (352,)
nifty origin prices: (403,)


In [3]:
# sanity check: these lengths must match forecasts.npy / nifty_forecasts.npy row counts
# (352 for Sensex, 403 for Nifty per the paper). If they don't match, context_len/step/horizon
# here don't match what was actually used to build forecasts.npy -- stop and check before continuing.
sensex_forecasts = np.load("../data/forecasts.npy")
nifty_forecasts = np.load("../data/nifty_forecasts.npy")

print("sensex:", sensex_origin_prices.shape[0], "vs", sensex_forecasts.shape[0], "forecast windows")
print("nifty: ", nifty_origin_prices.shape[0], "vs", nifty_forecasts.shape[0], "forecast windows")

assert sensex_origin_prices.shape[0] == sensex_forecasts.shape[0], "Sensex length mismatch -- check window params"
assert nifty_origin_prices.shape[0] == nifty_forecasts.shape[0], "Nifty length mismatch -- check window params"
print("lengths match, origin prices are correctly aligned with forecasts/actuals rows")

sensex: 352 vs 352 forecast windows
nifty:  403 vs 403 forecast windows
lengths match, origin prices are correctly aligned with forecasts/actuals rows


## Step 2: Load forecasts, actuals, and regime labels

Same arrays used throughout the paper's analysis (notebooks 09-11).

In [4]:
sensex_actuals = np.load("../data/actuals.npy")
sensex_regimes = np.load("../data/sensex_regime_labels.npy", allow_pickle=True)

nifty_actuals = np.load("../data/nifty_actuals.npy")
nifty_regimes = np.load("../data/nifty_regime_labels.npy", allow_pickle=True)

quantile_levels = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
LOWER_IDX = 1   # 10th percentile column
UPPER_IDX = 9   # 90th percentile column

## Step 3: Compute width in points AND as a percentage of origin price

This mirrors the Mechanism paragraph in Section 5.3/5.4 of the paper, which reported day-30 average
interval width in points for low-vol vs high-vol windows. We now compute both:
- `width_pts` -- exactly what the paper currently reports
- `width_pct` -- the new, price-level-normalized version: `(upper - lower) / origin_price`

In [5]:
def regime_width_comparison(forecasts, regimes, origin_prices, day, index_name):
    day_forecasts = forecasts[:, day, :]
    lower = day_forecasts[:, LOWER_IDX]
    upper = day_forecasts[:, UPPER_IDX]

    width_pts = upper - lower
    width_pct = width_pts / origin_prices * 100   # as a percentage

    low_mask = regimes == "low_vol"
    high_mask = regimes == "high_vol"

    print(f"--- {index_name}, day {day + 1} ahead ---")
    print(f"  low-vol  (n={low_mask.sum():3d}): {width_pts[low_mask].mean():8.1f} pts   |  {width_pct[low_mask].mean():5.2f}% of price")
    print(f"  high-vol (n={high_mask.sum():3d}): {width_pts[high_mask].mean():8.1f} pts   |  {width_pct[high_mask].mean():5.2f}% of price")

    pts_gap_relative = (width_pts[high_mask].mean() / width_pts[low_mask].mean() - 1) * 100
    pct_gap_relative = (width_pct[high_mask].mean() / width_pct[low_mask].mean() - 1) * 100
    print(f"  high-vol vs low-vol, points  : {pts_gap_relative:+.1f}%")
    print(f"  high-vol vs low-vol, percent : {pct_gap_relative:+.1f}%")
    print()

    return width_pts, width_pct

In [6]:
# Day 1 ahead (index 0) -- for reference, this is where the paper reports low-vol as better calibrated
_ = regime_width_comparison(sensex_forecasts, sensex_regimes, sensex_origin_prices, day=0, index_name="Sensex")
_ = regime_width_comparison(nifty_forecasts, nifty_regimes, nifty_origin_prices, day=0, index_name="Nifty")

--- Sensex, day 1 ahead ---
  low-vol  (n=197):    972.5 pts   |   2.09% of price
  high-vol (n=155):   1031.1 pts   |   2.77% of price
  high-vol vs low-vol, points  : +6.0%
  high-vol vs low-vol, percent : +32.2%

--- Nifty, day 1 ahead ---
  low-vol  (n=223):    282.4 pts   |   2.12% of price
  high-vol (n=180):    275.8 pts   |   2.84% of price
  high-vol vs low-vol, points  : -2.4%
  high-vol vs low-vol, percent : +33.7%



In [7]:
# Day 30 ahead (index 29) -- this is the headline comparison from the Mechanism paragraph:
# paper reports Sensex day-30 width as 4923.6 pts (low-vol) vs 5673.4 pts (high-vol), a +15.2% increase.
# check whether that +15.2% survives once we normalize by price level.
sensex_width_pts_30, sensex_width_pct_30 = regime_width_comparison(
    sensex_forecasts, sensex_regimes, sensex_origin_prices, day=29, index_name="Sensex"
)
nifty_width_pts_30, nifty_width_pct_30 = regime_width_comparison(
    nifty_forecasts, nifty_regimes, nifty_origin_prices, day=29, index_name="Nifty"
)

--- Sensex, day 30 ahead ---
  low-vol  (n=197):   4923.6 pts   |  10.66% of price
  high-vol (n=155):   5673.4 pts   |  15.31% of price
  high-vol vs low-vol, points  : +15.2%
  high-vol vs low-vol, percent : +43.7%

--- Nifty, day 30 ahead ---
  low-vol  (n=223):   1446.0 pts   |  10.85% of price
  high-vol (n=180):   1502.7 pts   |  15.77% of price
  high-vol vs low-vol, points  : +3.9%
  high-vol vs low-vol, percent : +45.3%



## Step 4: Interpreting the result

Compare the two "high-vol vs low-vol" percentage-gap numbers printed above (points vs percent) for day 30:

- **If they're similar** (e.g. both around +15%): the regime-conditional width difference is real and not
  a price-level artifact. Add a sentence to the paper's Mechanism paragraph confirming this and reporting
  the percentage-width numbers alongside the point numbers.
- **If the percentage gap is much smaller than the points gap** (e.g. points show +15% but percent shows
  only +3%): a meaningful chunk of the original finding was a price-level confound. This needs to be
  reported honestly -- soften the Mechanism claim and note the percentage-width numbers as the more
  reliable comparison.
- **If the percentage gap reverses sign**: this would be a serious finding worth flagging prominently
  rather than burying -- would mean the current "wider intervals in high-vol" story is essentially an
  artifact of when high-vol periods happened to occur in the sample.

## Optional: also check whether the *coverage reversal itself* is affected

Coverage (whether the actual value falls inside the interval) does NOT change under this kind of
normalization, since dividing both the interval bounds and the actual value by the same origin price
doesn't change whether the actual falls inside the interval. So Table 2 and Table 3 in the paper (coverage
numbers) don't need to be rerun -- only the width numbers in the Mechanism paragraph and Table on SARIMAX
width comparison (Section 5.4) are price-level-sensitive, since those are measured in absolute units.